# Day 02 · Explicit schemas and file ingestion

**Priority version:** 10 tasks that cover every unique concept of the day (all 5 assignments, 5 extra examples and 30 questions are mapped below). The last cell answers the conceptual questions.

| Task | Concept | Covers |
|---|---|---|
| 1 | Explicit schemas for all six tables | A1, A2, Q2, Q11, Q12, Q13 |
| 2 | Inferred vs explicit, leading zeros, decimal vs double | A3, Q1, Q5, Q14, Q27 |
| 3 | Tricky CSV values: quoted comma, blank vs `""`, header-only file | E1, E3, Q8, Q15, Q16 |
| 4 | Parse modes and corrupt records | A4, Q9, Q17, Q18, Q21, Q23, Q28 |
| 5 | `nullable=False` is not a data check | Q3, Q26 |
| 6 | Header order vs schema order | Q7, Q24 |
| 7 | Audit columns | A5, Q19 |
| 8 | Nested JSON events | E2, Q10, Q20 |
| 9 | Reusable reader: parse failures vs business-rule failures | Q25, Q29, Q30 |
| 10 | Quick extras: timezone assumption, a check that fails | E4, E5 |

Each task: a markdown cell (problem + expected result), then the solution cell. Run on the **pyspark-course** cluster, not Serverless.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
import json

RAW = "/Volumes/dbw_pyspark_course/retail/raw"
FIX = "/Volumes/dbw_pyspark_course/retail/scratch/day02"   # small test files we create today

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sql("CREATE VOLUME IF NOT EXISTS dbw_pyspark_course.retail.scratch")
dbutils.fs.mkdirs(FIX)

manifest = json.loads(open(f"{RAW}/manifest.json").read())

def write_fixture(name, text):
    """Write a small test file into the scratch volume and return its path."""
    path = f"{FIX}/{name}"
    dbutils.fs.put(path, text, overwrite=True)
    return path

### Task 1 · Explicit schemas for all six tables
**Covers:** A1, A2, Q2, Q11, Q12, Q13

**Problem:** Define a schema for each raw table, using the types from `DATA_DICTIONARY.md`:
- IDs as `STRING`
- money as `DECIMAL(12,2)` and `discount_pct` as `DECIMAL(5,2)`
- quantities as `INT`
- `signup_date` as `DATE`; `order_ts`, `return_ts` and `updated_at` as `TIMESTAMP` (UTC)

Read every table with its schema and compare the row counts with the manifest.

**Expected:**
- Counts: customers 80 · stores 5 · products 20 · orders 300 · order_items 785 · returns 30 (all equal to the manifest).
- `order_items` schema: `quantity: integer`, `unit_price: decimal(12,2)`, `discount_pct: decimal(5,2)`.
- `order_ts` is a real timestamp, shown as `2026-07-01 09:00:00`, not as text.

In [0]:
# StructType = the whole row; StructField = one column (name, type, nullable)
customers_schema = T.StructType([
    T.StructField("customer_id", T.StringType(),    False),
    T.StructField("name",        T.StringType(),    True),
    T.StructField("email",       T.StringType(),    True),
    T.StructField("city",        T.StringType(),    True),
    T.StructField("signup_date", T.DateType(),      True),
    T.StructField("updated_at",  T.TimestampType(), True),
])

# The same idea written as DDL strings: shorter, and Spark turns them into StructTypes
schemas = {
    "customers":   customers_schema,
    "stores":      "store_id STRING, store_name STRING, city STRING",
    "products":    "product_id STRING, product_name STRING, category STRING, list_price DECIMAL(12,2), cost_price DECIMAL(12,2)",
    "orders":      "order_id STRING, customer_id STRING, store_id STRING, order_ts TIMESTAMP, status STRING, channel STRING",
    "order_items": "line_id STRING, order_id STRING, product_id STRING, quantity INT, unit_price DECIMAL(12,2), discount_pct DECIMAL(5,2)",
    "returns":     "return_id STRING, line_id STRING, return_quantity INT, return_ts TIMESTAMP, reason STRING",
}

CSV_OPTS = {"header": True, "dateFormat": "yyyy-MM-dd", "timestampFormat": "yyyy-MM-dd'T'HH:mm:ssXXX"}

tables = {name: spark.read.options(**CSV_OPTS).schema(schema).csv(f"{RAW}/{name}.csv")
          for name, schema in schemas.items()}

for name, df in tables.items():
    print(f"{name:12} rows={df.count():4}  manifest={manifest['raw_counts'][name]:4}")

tables["order_items"].printSchema()
tables["orders"].show(3, truncate=False)

### Task 2 · Inferred vs explicit schemas
**Covers:** A3, Q1, Q5, Q14, Q27

**Problem:**
1. Read `order_items.csv` with `inferSchema=True` and compare its schema with your explicit one.
2. Read a file of store codes such as `007` with inference, then with an explicit `STRING` schema. Write the result out and read it back. Are the leading zeros kept?
3. Why use `DECIMAL` for money instead of `DOUBLE`?

**Expected:**
1. Inferred: `quantity: int`, `unit_price: int` (the file has `700`, not `700.00`), `discount_pct: double`. Inference also costs an **extra Spark job**, because Spark scans the file just to guess the types.
2. Inferred: `007` becomes `7` and `042` becomes `42`. The zeros are lost for good. With `STRING`, `007` survives the read, the write and the read back.
3. `0.1 + 0.2` as double is `0.30000000000000004`; as decimal it is exactly `0.30`. Money must not drift by fractions of a paisa.

In [0]:
# 1. Inferred vs explicit
inferred = (spark.read.option("header", True).option("inferSchema", True)
            .csv(f"{RAW}/order_items.csv"))          # extra job: Spark scans the file to guess types
print("inferred:", inferred.schema.simpleString())
print("explicit:", tables["order_items"].schema.simpleString())

# 2. Leading zeros (Q27)
codes_path = write_fixture("store_codes.csv", "store_code,pincode\n007,560001\n042,110001\n")
spark.read.option("header", True).option("inferSchema", True).csv(codes_path).show()

codes = spark.read.option("header", True).schema("store_code STRING, pincode STRING").csv(codes_path)
codes.write.mode("overwrite").option("header", True).csv(f"{FIX}/store_codes_out")
spark.read.option("header", True).schema("store_code STRING, pincode STRING").csv(f"{FIX}/store_codes_out").show()

# 3. Why DECIMAL for money (Q5)
spark.sql("""SELECT 0.1D + 0.2D AS as_double,
                    CAST(0.1 AS DECIMAL(12,2)) + CAST(0.2 AS DECIMAL(12,2)) AS as_decimal""").show()

### Task 3 · Tricky CSV values
**Covers:** E1, E3, Q8, Q15, Q16

**Problem:** Read a small fixture where:
- `C1`'s name contains a comma inside quotes: `"Sharma, Asha"`
- `C2`'s email is blank
- `C3`'s email is a quoted empty string `""`

Then read a header-only file with an explicit schema.

**Expected:**
- `C1` name = `Sharma, Asha`: the quotes protect the comma, so the row still has 3 columns.
- `C2` email = `null`, meaning "no value".
- `C3` email = `""`, meaning "a value that is empty text". It is **not** null.
- Header-only file: **0 rows**, but the 3 columns from the schema.

In [0]:
tricky_path = write_fixture("tricky.csv",
    'customer_id,name,email\n'
    'C1,"Sharma, Asha",asha@example.test\n'
    'C2,Ravi,\n'
    'C3,Meena,""\n')

tricky = spark.read.option("header", True).schema("customer_id STRING, name STRING, email STRING").csv(tricky_path)
tricky.select("customer_id", "name", "email",
              F.col("email").isNull().alias("email_is_null"),
              (F.col("email") == "").alias("email_is_empty_text")).show(truncate=False)

header_only_path = write_fixture("header_only.csv", "customer_id,name,email\n")
empty = spark.read.option("header", True).schema("customer_id STRING, name STRING, email STRING").csv(header_only_path)
print("rows:", empty.count(), "| columns:", empty.columns)

### Task 4 · Parse modes and corrupt records
**Covers:** A4, Q9, Q17, Q18, Q21, Q23, Q28

**Problem:** Use a 3-line fixture:
- `L1` is good.
- `L2` has quantity `two`, which is not a number.
- `L3` has unit_price `12345678901234.56`, too big for `DECIMAL(12,2)` (at most 10 digits before the point).

Read it in the three parse modes and compare.

**Expected:**
| Mode | Result |
|---|---|
| `PERMISSIVE` (default) | Keeps all 3 rows. The bad fields become `null`, and `_corrupt_record` holds the raw text of `L2` and `L3`. |
| `DROPMALFORMED` | `show()` gives only `L1`. But `count()` likely says **3**: Spark only parses the columns a query needs, and `count()` needs none, so nothing gets checked (column pruning, Q23). |
| `FAILFAST` | Building the DataFrame works, because it's lazy. The **action** fails with `MALFORMED_RECORD_IN_PARSING`. |

In [0]:
bad_path = write_fixture("bad_lines.csv",
    "line_id,quantity,unit_price\n"
    "L1,2,10.50\n"
    "L2,two,10.50\n"
    "L3,1,12345678901234.56\n")
line_schema = "line_id STRING, quantity INT, unit_price DECIMAL(12,2)"

# PERMISSIVE + corrupt-record column: keep every row, keep the raw text of bad ones
permissive = (spark.read.option("header", True)
              .option("mode", "PERMISSIVE")
              .option("columnNameOfCorruptRecord", "_corrupt_record")
              .schema(line_schema + ", _corrupt_record STRING")
              .csv(bad_path))
permissive.show(truncate=False)

In [0]:
# DROPMALFORMED: drop bad rows... but only the ones a query actually parses
dropped = spark.read.option("header", True).option("mode", "DROPMALFORMED").schema(line_schema).csv(bad_path)
dropped.show()
print("rows when all columns are read:", len(dropped.collect()))
print("count() with no columns needed:", dropped.count())

In [0]:
# FAILFAST: nothing fails until an action reads the data
failfast = spark.read.option("header", True).option("mode", "FAILFAST").schema(line_schema).csv(bad_path)
try:
    failfast.show()
except Exception as e:
    print(str(e)[:300])

### Task 5 · `nullable=False` is not a data check
**Covers:** Q3, Q26

**Problem:** Declare `order_id` as `NOT NULL` and read `order_items.csv`. Does Spark reject the two lines that have a blank `order_id`?

**Expected:** No. `printSchema()` shows `order_id: nullable = true`, because file readers relax every column to nullable. Rows `BAD08` and `BAD09` come through with a null `order_id`. Nullability is a hint for the optimizer, not a data-quality check. Real checks need explicit filters and quarantine (Day 04).

In [0]:
strict = (spark.read.options(**CSV_OPTS)
          .schema("line_id STRING NOT NULL, order_id STRING NOT NULL, product_id STRING, "
                  "quantity INT, unit_price DECIMAL(12,2), discount_pct DECIMAL(5,2)")
          .csv(f"{RAW}/order_items.csv"))
strict.printSchema()
strict.filter(F.col("order_id").isNull()).show()

### Task 6 · Header order vs schema order
**Covers:** Q7, Q24

**Problem:** A file arrives with its columns in a different order (`name,customer_id`) from your schema (`customer_id, name`). What happens?

**Expected:**
- With an explicit schema, CSV columns are matched by **position**, not by header name. `customer_id` silently gets `Asha` and `Ravi`; there is only a warning in the driver log.
- `enforceSchema=false` makes Spark compare the header with the schema, and it fails: `CSV header does not conform to the schema`.
- For the other Q24 cases: a **missing column** gives rows that are short of fields, which are malformed and padded with nulls. An **incompatible type** gives null plus `_corrupt_record` (Task 4).

In [0]:
reordered_path = write_fixture("reordered.csv", "name,customer_id\nAsha,C001\nRavi,C002\n")
schema2 = "customer_id STRING, name STRING"

spark.read.option("header", True).schema(schema2).csv(reordered_path).show()   # silently swapped

try:
    (spark.read.option("header", True).option("enforceSchema", False)
     .schema(schema2).csv(reordered_path).show())
except Exception as e:
    print(str(e)[:300])

### Task 7 · Audit columns
**Covers:** A5, Q19

**Problem:** Add the source file path and the ingestion time to every row of `orders`.

**Expected:** Every row has `source_file` = `.../raw/orders.csv` and the same `ingested_at` time for this run. `_metadata` is a hidden column that every file-based DataFrame has (Spark 3.3+). On Unity Catalog, use it instead of the older `input_file_name()`.

In [0]:
orders_audit = tables["orders"].select(
    "*",
    F.col("_metadata.file_path").alias("source_file"),
    F.current_timestamp().alias("ingested_at"),
)
orders_audit.select("order_id", "source_file", "ingested_at").show(3, truncate=False)

### Task 8 · Nested JSON events
**Covers:** E2, Q10, Q20

**Before running:** go to **Catalog → dbw_pyspark_course → retail → raw** → **Create directory** `events_archive`, then upload `data/events_archive/batch_001.json` into it. You can upload all 6 batch files now; the streaming days use them.

**Problem:** Read one event file using the nested `event_schema` from `examples/schemas.py`, and pull out a few nested fields.

**Expected:** 8 events: 3 `view` events with 0 items and 5 `purchase` events with 2 items each. `device.os` = `android`, `attributes['campaign']` = `autumn`, and `event_time` is a real timestamp.

In [0]:
# Copied from examples/schemas.py
event_schema = T.StructType([
    T.StructField("event_id", T.StringType()),
    T.StructField("event_time", T.TimestampType()),
    T.StructField("customer_id", T.StringType()),
    T.StructField("order_id", T.StringType()),
    T.StructField("event_type", T.StringType()),
    T.StructField("items", T.ArrayType(T.StructType([
        T.StructField("product_id", T.StringType()),
        T.StructField("quantity", T.IntegerType()),
        T.StructField("unit_price", T.DecimalType(12, 2)),
    ]))),
    T.StructField("device", T.StructType([
        T.StructField("os", T.StringType()), T.StructField("app_version", T.StringType()),
    ])),
    T.StructField("attributes", T.MapType(T.StringType(), T.StringType())),
])

events = spark.read.schema(event_schema).json(f"{RAW}/events_archive/batch_001.json")
events.printSchema()
events.select(
    "event_id", "event_time", "event_type",
    F.size("items").alias("n_items"),                    # array length
    F.col("device.os").alias("os"),                      # struct field
    F.col("attributes")["campaign"].alias("campaign"),   # map lookup
).show(truncate=False)

### Task 9 · Reusable reader: parse failures vs business-rule failures
**Covers:** Q25, Q29, Q30 · completion check "a malformed row is visible in your audit output"

**Problem:** Write `read_csv(path, schema, mode, audit)` that applies an explicit schema, keeps a `_corrupt_record` column in PERMISSIVE mode, and adds audit columns. Use it to separate two kinds of bad rows:
- **Parse failures:** the text could not be typed at all. Quarantine them and keep the raw text.
- **Business-rule failures:** the row typed fine, but its values break a rule.

**Expected:**
- Fixture: `L2` and `L3` appear as parse failures, with their raw text and `source_file`.
- Real `order_items`: **0** parse failures and **8** business-rule failures: `BAD00`–`BAD02` (quantity −1), `BAD03`–`BAD05` (discount 1.50) and `BAD08`–`BAD09` (missing order_id). `BAD06` and `BAD07` have an unknown product, which needs a join (Day 04).

**Gotcha:** Spark refuses a query on a raw file that reads **only** `_corrupt_record`, for example `.filter(...corrupt...).count()`. Show whole rows as below, or `cache()` the DataFrame first.

In [0]:
def read_csv(path, schema, mode="PERMISSIVE", audit=True):
    """Reusable reader. schema: DDL string. Adds _corrupt_record (PERMISSIVE) and audit columns."""
    if mode == "PERMISSIVE":
        schema = schema + ", _corrupt_record STRING"     # keeps the raw text of lines that can't be typed
    df = (spark.read.options(**CSV_OPTS)
          .option("mode", mode)
          .option("columnNameOfCorruptRecord", "_corrupt_record")
          .schema(schema)
          .csv(path))
    if audit:
        df = df.select("*", F.col("_metadata.file_path").alias("source_file"),
                       F.current_timestamp().alias("ingested_at"))
    return df

# Parse failures: the line could not be typed -> quarantine it with its raw text
fixture_rows = read_csv(bad_path, line_schema)
fixture_rows.filter(F.col("_corrupt_record").isNotNull()).show(truncate=False)

# Real order_items: every line parses, but some values break business rules
items = read_csv(f"{RAW}/order_items.csv", schemas["order_items"])
items.filter(F.col("_corrupt_record").isNotNull()).show()      # expected: empty

business_failures = items.filter(
    F.col("order_id").isNull() | (F.col("quantity") <= 0) | (F.col("discount_pct") > 1))
business_failures.select("line_id", "order_id", "quantity", "discount_pct").show()
print("business-rule failures:", business_failures.count())

### Task 10 · Quick extras
**Covers:** E4, E5

| | Problem | Expected |
|---|---|---|
| a | **Change an assumption (E4):** "every timestamp carries an offset (`Z`)". What if one arrives as `2026-07-01 09:00:00` with no offset? | It is read in the **session timezone**, so the same text becomes a different instant: UTC → `1782896400`, Asia/Kolkata → `1782876600` (5 h 30 min apart) |
| b | **A check that fails when the code is broken (E5):** `order_id` and `order_ts` must never be null. Break it by swapping two columns in the schema | Correct schema passes. The broken schema fails with `order_ts: 300` nulls, because positions matter (Task 6) |

In [0]:
# (a) E4 · a timestamp without an offset is read in the session timezone
no_offset_path = write_fixture("no_offset.csv", "order_id,order_ts\nO1,2026-07-01 09:00:00\n")
for tz in ["UTC", "Asia/Kolkata"]:
    spark.conf.set("spark.sql.session.timeZone", tz)
    epoch = (spark.read.option("header", True).schema("order_id STRING, order_ts TIMESTAMP")
             .csv(no_offset_path).select(F.unix_timestamp("order_ts")).first()[0])
    print(f"{tz:13} -> epoch seconds {epoch}")
spark.conf.set("spark.sql.session.timeZone", "UTC")

# (b) E5 · a check that must fail when the code is broken
def assert_no_nulls(df, cols):
    nulls = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]).first().asDict()
    assert all(v == 0 for v in nulls.values()), f"unexpected nulls: {nulls}"

assert_no_nulls(tables["orders"], ["order_id", "order_ts"])
print("correct schema: passed")

broken_schema = "order_id STRING, order_ts TIMESTAMP, store_id STRING, customer_id STRING, status STRING, channel STRING"
broken = spark.read.options(**CSV_OPTS).schema(broken_schema).csv(f"{RAW}/orders.csv")
try:
    assert_no_nulls(broken, ["order_id", "order_ts"])
except AssertionError as e:
    print("broken schema: failed as expected ->", e)

### Concept notes (Q1–Q7, Q9, Q10, Q22, Q26): read once, then rewrite in your own words
- **Why inference is risky for IDs (Q1):** Spark guesses from the values. `007` becomes the number 7, a column can change type from one file to the next, and guessing costs an extra pass over the data.
- **StructType / StructField (Q2):** a `StructType` is the shape of a whole row; each `StructField` is one column with its name, type and nullable flag.
- **nullable (Q3):** whether a column may hold null. Reading files does **not** enforce it (Task 5).
- **Quantity (Q4):** `INT`, because quantities are whole units. Use `LONG` only if counts could exceed about 2.1 billion.
- **Decimal for prices (Q5):** exact base-10 arithmetic, so totals don't drift (Task 2).
- **DATE vs TIMESTAMP (Q6):** `DATE` is a calendar day with no time. `TIMESTAMP` is an exact instant, displayed in the session timezone.
- **header option (Q7):** uses the first line as column names, and skips it as data. With an explicit schema, columns are still matched by **position** (Task 6).
- **PERMISSIVE (Q9):** keeps every row. Fields that can't be typed become null, and the raw line can be kept in `_corrupt_record`.
- **Streams need a schema (Q10):** a stream starts before all its files exist, so there is nothing reliable to infer from. The schema must also stay fixed for the whole life of the query and its checkpoint.
- **Fewer or extra tokens aren't always flagged (Q22):** Spark parses only the columns a query needs (column pruning), so a short or long line may never be checked. Without a `_corrupt_record` column, PERMISSIVE just fills nulls silently.
- **nullable=False isn't enough (Q26):** readers ignore it, and it can't express business rules such as "quantity > 0". You need explicit checks plus a quarantine (Day 04, Day 18).